# SPIDER — Notebook 4: learning curves

**Question.** At what training-set size does a fine-tuned CNN overtake five
explicit measurements — and does the crossover depend on whether the grade is
defined geometrically?

**Design.**

* Outer patient-grouped 5-fold split, stratified on each target — the test folds
  never change, so every point on a curve is scored on the same patients.
* Inside each training fold, patients are shuffled once per *draw* and the first
  5%, 10%, 25%, 50% or 100% are kept. Subsets are **nested**: the 10% set contains
  the 5% set, and so on. This removes most of the noise that independent
  subsamples would add between neighbouring points.
* Sampling is by **patient**, never by disc, so no spine is split across train
  and test at any fraction.
* **Equal optimisation budget for the CNN.** A fixed number of epochs would give
  the 5% CNN about a tenth of the gradient steps of the 100% CNN and handicap it
  unfairly. Every CNN run instead gets at least `MIN_STEPS` updates.
* The 10% point is roughly the scale of BUU-LSPINE (59 positive patients),
  which is what lets the two studies be read on one axis.

**Arms.** Geometry and DINOv2 probes on all four targets (seconds). The CNN on
two contrasting targets by default — narrowing, where it tied geometry, and
Pfirrmann >= 4, where it won by 0.074. About 50 minutes on a T4, checkpointed.

## Cell 1 — Load and configure

In [ ]:
import os, time, math, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/spider'
CROP_TAG = 'tight'

prim = pd.read_csv(f'{DRIVE}/spider_primary_cohort.csv')
z = np.load(f'{DRIVE}/spider_crops_{CROP_TAG}.npz', allow_pickle=True)
X_img, row_index = z['images'], z['row_index']
E = np.load(f'{DRIVE}/spider_dinov2_{CROP_TAG}.npy')
prim = prim.loc[row_index].reset_index(drop=True)
assert len(prim) == len(E) == len(X_img)
print(f"aligned: {len(prim)} discs, {prim.Patient.nunique()} patients")

INV = ['disc_height_ratio', 'ap_ratio', 'canal_ratio', 'volume_ratio', 'slip_pct']
TARGETS = {'disc narrowing': 'narrowing', 'disc bulging': 'bulging',
           'Pfirrmann >= 4': 'pfirrmann_severe', 'Modic (any)': 'modic_any'}

FRACTIONS = [0.05, 0.10, 0.25, 0.50, 1.00]
N_DRAWS = 2                                 # independent patient orderings
CNN_TARGETS = ['narrowing', 'pfirrmann_severe']
MIN_PER_LEVEL = 8                           # per-level z-score needs this many discs
SEED = 0
LC = f'{DRIVE}/nb4_learning_curve'
os.makedirs(LC, exist_ok=True)

## Cell 2 — Splits, nested subsamples, and the three arms

`MIN_PER_LEVEL` is lowered to 8 here (it was 20 in Notebook 3). At 5% of the
training data there are only about nine discs per level; with the old threshold
every level would fall back to global statistics, silently removing the per-level
normalisation that Notebook 1 showed the geometry depends on. At 100% the counts
are far above either threshold, so the full-data points are unaffected.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

levels = prim['ivd_label'].values
pats = prim['Patient'].values


def outer_splits(y):
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    return list(sgkf.split(np.zeros(len(y)), y, pats))


def subsample(tr, frac, draw, fold):
    # nested: one fixed patient ordering per (draw, fold); keep the first frac of it
    up = np.unique(pats[tr])
    order = np.random.default_rng(10_000 * draw + fold).permutation(up)
    keep = order[:max(2, math.ceil(frac * len(up)))]
    return tr[np.isin(pats[tr], keep)]


def _zscore(ref, ref_lvl, X, lvl):
    out = X.copy()
    gm, gs = ref.mean(), ref.std().replace(0, 1)
    for l in np.unique(lvl):
        m = ref_lvl == l
        ok = m.sum() >= MIN_PER_LEVEL
        mu = ref[m].mean() if ok else gm
        sd = ref[m].std().replace(0, 1) if ok else gs
        out.loc[lvl == l] = (X.loc[lvl == l] - mu) / sd
    return out.fillna(0)


def _lr():
    return LogisticRegression(max_iter=5000, class_weight='balanced',
                              random_state=SEED)


def fit_geometry(tr, te, y):
    A, B = prim[INV].iloc[tr], prim[INV].iloc[te]
    A2 = _zscore(A, levels[tr], A, levels[tr])
    B2 = _zscore(A, levels[tr], B, levels[te])
    m = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(),
                      _lr()).fit(A2, y[tr])
    return m.predict_proba(B2)[:, 1]


def fit_dinov2(tr, te, y):
    k = max(2, min(128, len(tr) - 1))
    m = make_pipeline(StandardScaler(), PCA(n_components=k, random_state=SEED),
                      _lr()).fit(E[tr], y[tr])
    return m.predict_proba(E[te])[:, 1]


def pooled_oof(fit, y, splits, frac, draw):
    # one prediction per disc; NaN where a fold's subsample had a single class
    p = np.full(len(y), np.nan)
    npos, ntr = [], []
    for k, (tr, te) in enumerate(splits):
        sub = subsample(tr, frac, draw, k)
        if len(np.unique(y[sub])) < 2:
            continue
        p[te] = fit(sub, te, y)
        npos.append(int(y[sub].sum())); ntr.append(len(sub))
    return p, np.mean(npos) if npos else 0, np.mean(ntr) if ntr else 0


def score(y, p):
    k = ~np.isnan(p)
    if k.sum() < 20 or len(np.unique(y[k])) < 2:
        return np.nan, np.nan
    return roc_auc_score(y[k], p[k]), average_precision_score(y[k], p[k])

print("ready")

## Cell 3 — Geometry and DINOv2 curves (all four targets, under a minute)

In [ ]:
curves, oof = [], {}
t0 = time.time()
for tname, tcol in TARGETS.items():
    y = prim[tcol].astype(int).values
    splits = outer_splits(y)
    for frac in FRACTIONS:
        for draw in range(N_DRAWS if frac < 1 else 1):
            for arm, fit in (('geometry', fit_geometry), ('dinov2', fit_dinov2)):
                p, npos, ntr = pooled_oof(fit, y, splits, frac, draw)
                auc, ap = score(y, p)
                oof[(tcol, arm, frac, draw)] = p
                curves.append(dict(target=tname, tcol=tcol, arm=arm, frac=frac,
                                   draw=draw, train_discs=ntr, train_pos=npos,
                                   auc=auc, pr_auc=ap))
    print(f"  {tname:16s} done  ({time.time()-t0:.0f}s)", flush=True)

curves = pd.DataFrame(curves)
curves.to_csv(f'{LC}/curves_geom_dino.csv', index=False)
np.save(f'{LC}/oof_geom_dino.npy', oof, allow_pickle=True)

print("\nmean AUC by training fraction")
print(curves.groupby(['target', 'arm', 'frac']).auc.mean()
      .unstack('frac').round(3).to_string())

## Cell 4 — Fine-tuned CNN curves (checkpointed)

Each (target, fraction, draw) is saved to Drive as soon as its five folds finish.
If Colab disconnects or the GPU quota runs out, re-run Cells 1-3 and this cell:
completed runs are skipped. Roughly 50 minutes for the two default targets.

In [ ]:
import torch, torch.nn as nn, torchvision
from torch.utils.data import TensorDataset, DataLoader

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
assert dev == 'cuda', "switch to a T4 GPU: Runtime > Change runtime type"
Xt = torch.from_numpy(X_img).float().div(255).unsqueeze(1).repeat(1, 3, 1, 1)
EPOCHS, MIN_STEPS, BATCH = 8, 200, 32


def fit_cnn_factory(seed):
    def fit(tr, te, y):
        torch.manual_seed(seed); np.random.seed(seed)
        net = torchvision.models.efficientnet_b0(weights='IMAGENET1K_V1')
        net.classifier[1] = nn.Linear(net.classifier[1].in_features, 1)
        net = net.to(dev)
        g = torch.Generator().manual_seed(seed)
        dl = DataLoader(TensorDataset(Xt[tr], torch.tensor(y[tr]).float()),
                        batch_size=BATCH, shuffle=True, generator=g)
        # equal optimisation budget: at least MIN_STEPS updates at every size
        total = max(EPOCHS * len(dl), MIN_STEPS)
        pw = torch.tensor([(y[tr] == 0).sum() / max((y[tr] == 1).sum(), 1)],
                          dtype=torch.float32).to(dev)
        lossf = nn.BCEWithLogitsLoss(pos_weight=pw)
        opt = torch.optim.AdamW(net.parameters(), lr=3e-4, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total)
        net.train(); step = 0
        while step < total:
            for xb, yb in dl:
                opt.zero_grad()
                lossf(net(xb.to(dev)).squeeze(1), yb.to(dev)).backward()
                opt.step(); sched.step(); step += 1
                if step >= total:
                    break
        net.eval(); out = []
        with torch.no_grad():
            for i in range(0, len(te), 64):
                out.append(torch.sigmoid(
                    net(Xt[te[i:i+64]].to(dev)).squeeze(1)).cpu().numpy())
        del net; torch.cuda.empty_cache()
        return np.concatenate(out)
    return fit


t_all = time.time()
for tcol in CNN_TARGETS:
    tname = [k for k, v in TARGETS.items() if v == tcol][0]
    y = prim[tcol].astype(int).values
    splits = outer_splits(y)
    for fi, frac in enumerate(FRACTIONS):
        for draw in range(N_DRAWS if frac < 1 else 1):
            f = f'{LC}/cnn_{tcol}_f{int(frac*100):03d}_d{draw}.npz'
            if os.path.exists(f):
                print(f"  {tname:16s} {frac:4.0%} draw {draw}  cached", flush=True)
                continue
            t0 = time.time()
            p = np.full(len(y), np.nan); npos, ntr = [], []
            for k, (tr, te) in enumerate(splits):
                sub = subsample(tr, frac, draw, k)
                if len(np.unique(y[sub])) < 2:
                    continue
                p[te] = fit_cnn_factory(10_000 * draw + 100 * fi + k)(sub, te, y)
                npos.append(int(y[sub].sum())); ntr.append(len(sub))
            np.savez(f, p=p, train_pos=np.mean(npos), train_discs=np.mean(ntr))
            print(f"  {tname:16s} {frac:4.0%} draw {draw}  AUC {score(y, p)[0]:.3f}"
                  f"  ({time.time()-t0:.0f}s, total {(time.time()-t_all)/60:.1f} min)",
                  flush=True)
print("CNN curves complete")

## Cell 5 — Tables and the crossover point

In [ ]:
rows = []
for tcol in CNN_TARGETS:
    tname = [k for k, v in TARGETS.items() if v == tcol][0]
    y = prim[tcol].astype(int).values
    for frac in FRACTIONS:
        for draw in range(N_DRAWS if frac < 1 else 1):
            f = f'{LC}/cnn_{tcol}_f{int(frac*100):03d}_d{draw}.npz'
            if not os.path.exists(f):
                continue
            d = np.load(f)
            oof[(tcol, 'cnn', frac, draw)] = d['p']
            auc, ap = score(y, d['p'])
            rows.append(dict(target=tname, tcol=tcol, arm='cnn', frac=frac, draw=draw,
                             train_discs=float(d['train_discs']),
                             train_pos=float(d['train_pos']), auc=auc, pr_auc=ap))
allc = pd.concat([curves, pd.DataFrame(rows)], ignore_index=True)
allc.to_csv(f'{LC}/curves_all_arms.csv', index=False)

summ = (allc.groupby(['target', 'arm', 'frac'])
            .agg(train_pos=('train_pos', 'mean'), auc=('auc', 'mean'),
                 auc_sd=('auc', 'std'), pr_auc=('pr_auc', 'mean'))
            .reset_index())

for tname in TARGETS:
    s = summ[summ.target == tname]
    if s.empty:
        continue
    print(f"\n{tname}")
    t = s.pivot(index='frac', columns='arm', values='auc')
    t.insert(0, 'train positives', s.groupby('frac').train_pos.mean().round(0))
    print(t.round(3).to_string())

print("\n" + "=" * 60)
print("CROSSOVER: smallest fraction at which the CNN's mean AUC exceeds geometry's")
for tcol in CNN_TARGETS:
    tname = [k for k, v in TARGETS.items() if v == tcol][0]
    s = summ[summ.target == tname].pivot(index='frac', columns='arm', values='auc')
    if 'cnn' not in s:
        continue
    ahead = s.index[(s['cnn'] > s['geometry']).values]
    pos = summ[(summ.target == tname) & (summ.arm == 'geometry')].set_index('frac').train_pos
    if len(ahead):
        f0 = ahead.min()
        print(f"  {tname:16s} CNN ahead from {f0:.0%} of training data "
              f"(~{pos[f0]:.0f} positive training discs)")
    else:
        print(f"  {tname:16s} CNN never ahead of geometry")
    below = s.index[(s['cnn'] <= s['geometry']).values]
    if len(below):
        print(f"  {'':16s} geometry at or ahead at: "
              f"{', '.join(f'{b:.0%}' for b in below)}")

## Cell 6 — The figure

x-axis: positive training discs, log scale — the quantity that actually limits a
classifier, and the one that lets BUU-LSPINE be placed on the same axis. The
dashed line marks BUU-LSPINE's 59 positives. Note the unit differs (patients
there, discs here), so the line is an order-of-magnitude marker, not an exact
equivalence.

In [ ]:
import matplotlib.pyplot as plt
colors = {'geometry': '#1f77b4', 'dinov2': '#ff7f0e', 'cnn': '#2ca02c'}
fig, axes = plt.subplots(1, 4, figsize=(18, 4.2), sharey=True)
for ax, tname in zip(axes, TARGETS):
    s = summ[summ.target == tname]
    for arm in ['geometry', 'dinov2', 'cnn']:
        a = s[s.arm == arm].sort_values('frac')
        if a.empty:
            continue
        ax.errorbar(a.train_pos, a.auc, yerr=a.auc_sd.fillna(0), marker='o',
                    ms=4, capsize=2, color=colors[arm], label=arm)
    ax.axvline(59, ls='--', lw=.8, color='grey')
    ax.text(59, ax.get_ylim()[0] if False else 0.52, ' BUU-LSPINE\n scale',
            fontsize=7, color='grey', va='bottom')
    ax.set_xscale('log'); ax.set_title(tname, fontsize=10)
    ax.set_xlabel('positive training discs (log)')
    ax.grid(alpha=.3)
axes[0].set_ylabel('pooled out-of-fold AUC'); axes[0].legend(fontsize=8)
axes[0].set_ylim(0.5, 0.95)
plt.tight_layout()
plt.savefig(f'{LC}/learning_curves.png', dpi=200)
plt.savefig(f'{LC}/learning_curves.pdf')
plt.show()

## Cell 7 — Is the CNN - geometry gap real at each size?

Paired patient-level bootstrap on the pooled out-of-fold predictions, pooled
across draws, with Benjamini-Hochberg across every (target, fraction) test.

In [ ]:
N_BOOT = 1000
rng = np.random.default_rng(SEED)
uniq = np.unique(pats)
boot = [np.concatenate([np.flatnonzero(pats == u)
                        for u in rng.choice(uniq, len(uniq), replace=True)])
        for _ in range(N_BOOT)]

rows = []
for tcol in CNN_TARGETS:
    tname = [k for k, v in TARGETS.items() if v == tcol][0]
    y = prim[tcol].astype(int).values
    for frac in FRACTIONS:
        dist, point = [], []
        for draw in range(N_DRAWS if frac < 1 else 1):
            if (tcol, 'cnn', frac, draw) not in oof:
                continue
            pc, pg = oof[(tcol, 'cnn', frac, draw)], oof[(tcol, 'geometry', frac, draw)]
            k = ~(np.isnan(pc) | np.isnan(pg))
            point.append(roc_auc_score(y[k], pc[k]) - roc_auc_score(y[k], pg[k]))
            for i in boot:
                i = i[k[i]]
                if len(np.unique(y[i])) > 1:
                    dist.append(roc_auc_score(y[i], pc[i]) - roc_auc_score(y[i], pg[i]))
        if not dist:
            continue
        dist = np.array(dist)
        lo, hi = np.percentile(dist, [2.5, 97.5])
        pval = 2 * min((dist <= 0).mean(), (dist >= 0).mean())
        rows.append(dict(target=tname, frac=frac, dauc=np.mean(point),
                         lo=lo, hi=hi, p=max(pval, 1 / len(dist))))

sig = pd.DataFrame(rows).sort_values('p').reset_index(drop=True)
adj = sig.p * len(sig) / (sig.index + 1)
sig['p_bh'] = adj[::-1].cummin()[::-1].clip(upper=1)
sig['verdict'] = np.where(sig.p_bh >= .05, 'indistinguishable',
                          np.where(sig.dauc > 0, 'CNN better', 'geometry better'))
sig = sig.sort_values(['target', 'frac'])
print("CNN minus geometry AUC, by training fraction (BH across all rows)")
print("=" * 78)
print(sig.round(4).to_string(index=False))
sig.to_csv(f'{LC}/significance_by_fraction.csv', index=False)
print(f"\nall outputs in {LC}")